In [3]:
import pandas as pd
import numpy as np
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

pd.set_option('display.max_columns', None)  

In [42]:
def read_Bloomberg_port(fn):
  df_2023 = pd.read_excel(fn, skiprows=2)
  col_mapping = {
      'Ticker': 'Ticker',
      'Short Name': 'ShortName',
      'PORT US Sz Fact Exp:D-1': 'Size',
      'PORT US Prof Fact Exp:D-1': 'Profitability',
      'PORT US Vol Fact Exp:D-1': 'Volatility',
      'PORT US Trd Act Fact Exp:D-1': 'TradingActivity',
      'PORT US Lev Fact Exp:D-1': 'Leverage',
      'PORT US Mom Fact Exp:D-1': 'Momentum',
      'PORT US Val Fact Exp:D-1': 'Value',
      'PORT US Gr Fact Exp:D-1': 'Growth',
      'PORT US Dvd Yld Fact Exp:D-1': 'DividendYield',
      'PORT US Earn Var Fact Exp:D-1': 'EarningsVariation',
      'P/S': 'PriceToSales',
      'BEst P/S BF12M': 'EstPriceToSales',
      'P/B': 'PriceToBook',
      'BEst P/B BF12M': 'EstPriceToBook',
      'P/E': 'PriceToEarnings',
      'BEst P/E BF12M': 'EstPriceToEarnings',
      'ROE LF': 'ReturnOnEquity',
      'T12M R&D/Sls': 'RnDToSales',
      'GICS Sector': 'Sector',
      'GICS Ind': 'Industry',
      'GICS SubInd': 'SubIndustry',
      'GICS Sector.1': 'SectorName',
      'GICS Ind Name': 'IndustryName',
      'GICS SubInd Name': 'SubIndustryName',
      'ESG Disclosure Score (Latest Available) (BLOOMBERG L.P.)': 'ESGScore',
      'Beta:Y-1': 'Beta',
      'Total Return:Y-1': 'Return1Yr',
      'Number of Employees:Y': 'NumberOfEmployees',
      'Number of Employees:Y-5': 'NumberOfEmployees5YearsAgo'
  }
  df_2023 = df_2023.loc[~df_2023['Short Name'].isna()]
  df_2023.rename(columns=col_mapping, inplace=True)
  df_2023.drop(columns=['Ticker.1'], inplace=True)
  df_2023.drop(df_2023[df_2023['ShortName'].str.contains('ADR')].index, inplace=True)
  df_2023.Ticker=df_2023.Ticker.map(lambda s: s.split(' ')[0])
  df_2023 = df_2023.loc[df_2023.Ticker.map(len)<=4]
  return df_2023

df_2023 = read_Bloomberg_port('data/original_data/20230101_SimplePort.xlsx')
df_2024 = read_Bloomberg_port('data/original_data/20240101_SimplePort.xlsx')
df_2023_tmp = read_Bloomberg_port('data/original_data/20230101_SimplePort.xlsx')
sec2sec_name = df_2023_tmp.set_index('Sector')['SectorName'].to_dict()
secname2sec = df_2023_tmp.set_index('SectorName')['Sector'].to_dict()

In [43]:
df_2024.columns

Index(['Ticker', 'ShortName', 'Size', 'Profitability', 'Volatility',
       'TradingActivity', 'Leverage', 'Momentum', 'Value', 'Growth',
       'DividendYield', 'EarningsVariation', 'PriceToSales', 'EstPriceToSales',
       'PriceToBook', 'EstPriceToBook', 'PriceToEarnings',
       'EstPriceToEarnings', 'ReturnOnEquity', 'RnDToSales', 'Sector',
       'Industry', 'SubIndustry', 'SectorName', 'IndustryName',
       'SubIndustryName', 'ESGScore', 'Beta', 'Return1Yr', 'NumberOfEmployees',
       'NumberOfEmployees5YearsAgo'],
      dtype='object')

In [49]:
df_2024

,Ticker,ShortName,Size,Profitability,Volatility,TradingActivity,Leverage,Momentum,Value,Growth,DividendYield,EarningsVariation,PriceToSales,EstPriceToSales,PriceToBook,EstPriceToBook,PriceToEarnings,EstPriceToEarnings,ReturnOnEquity,RnDToSales,Sector,Industry,SubIndustry,SectorName,IndustryName,SubIndustryName,ESGScore,Beta,Return1Yr,NumberOfEmployees,NumberOfEmployees5YearsAgo
4,AAL,AMERICAN AIRLINE,-0.872789,-0.884676,-0.961996,3.000228,2.427836,-0.610635,2.050937,-0.812962,-0.722812,0.138502,0.169481,0.163585,NaN,NaN,4.043788,6.773244,NaN,NaN,20.0,203020.0,20302010.0,Industrials,Passenger Airlines,Passenger Airlines,65.878685,1.227147,8.188976,129700.0,126600.0
5,AAPL,APPLE INC,1.973204,2.738760,0.495583,-0.478149,0.085498,-0.130112,-0.372220,0.031300,-0.472168,0.460963,7.914328,7.466528,48.174509,40.067432,31.459150,28.716615,171.949509,8.071053,45.0,452020.0,45202030.0,Information Technology,"Technology Hardware, Storage & Peripherals","Technology Hardware, Storage & Peripherals",57.318176,1.075160,49.371970,161000.0,132000.0
8,ABBV,ABBVIE INC,0.496940,1.530953,0.124997,-0.728043,0.839538,-0.148440,0.063431,-0.557802,1.094617,-0.410797,4.976616,5.115141,22.782770,18.597456,18.267769,13.823277,46.069496,22.705648,35.0,352010.0,35201010.0,Health Care,Biotechnology,Biotechnology,69.017204,0.468575,-0.810278,50000.0,29000.0
11,ABNB,AIRBNB INC-A,-0.819412,0.108318,-0.543316,1.690136,-0.697833,-0.036284,-0.447422,1.276618,-0.722812,1.897143,9.023584,8.042780,9.669924,7.524665,16.481900,29.302311,74.472473,18.519186,25.0,253010.0,25301020.0,Consumer Discretionary,"Hotels, Restaurants & Leisure","Hotels, Resorts & Cruise Lines",42.510815,1.466884,59.732482,6811.0,NaN
12,ABT,ABBOTT LABS,0.098690,-0.457890,-0.388962,-0.784018,-0.477903,-0.687410,-0.191666,-0.600183,0.244200,-0.375825,4.803144,4.586008,5.098262,4.964078,33.691698,23.822355,14.112308,6.779499,35.0,351010.0,35101010.0,Health Care,Health Care Equipment & Supplies,Health Care Equipment,79.629814,0.718053,1.781966,115000.0,99000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684,WMT,WALMART INC,1.390133,-0.755175,-0.034558,-0.883312,-0.200955,0.060899,0.007342,-0.478348,-0.080722,-0.324086,0.664872,0.638207,5.343227,4.635032,23.958965,22.375866,21.477961,0,30.0,301010.0,30101040.0,Consumer Staples,Consumer Staples Distribution & Retail,Consumer Staples Merchandise Retail,54.471382,0.591213,12.599971,2100000.0,2300000.0
687,XEL,XCEL ENERGY INC,-0.731118,-0.785769,-0.226984,-0.110820,1.115379,-1.180554,0.434117,-0.838571,0.942531,-0.913837,2.300985,2.172828,1.973161,1.829384,19.304255,17.309397,10.334491,NaN,55.0,551010.0,55101010.0,Utilities,Electric Utilities,Electric Utilities,71.884117,0.673637,-9.926571,11982.0,11075.0
690,XOM,EXXON MOBIL CORP,1.375566,-0.080958,0.202097,-0.457229,-0.826035,0.131019,0.967992,-0.498213,0.757364,0.519116,1.179272,1.146779,1.984050,1.730857,9.590236,10.817643,21.321762,NaN,10.0,101020.0,10102010.0,Energy,"Oil, Gas & Consumable Fuels",Integrated Oil & Gas,66.230759,0.696742,-5.299010,62000.0,69600.0
691,XYZ,BLOCK INC,-0.932959,-1.413631,-0.066629,3.000228,-0.566332,-1.311343,-0.249530,1.817979,-0.722812,0.570095,2.233461,1.926434,2.661211,2.421789,NaN,24.559298,-0.973276,12.082453,40.0,402010.0,40201060.0,Financials,Financial Services,Transaction & Payment Processing Services,45.991348,1.777374,22.933885,12428.0,2338.0


In [35]:
df_2024[["Ticker",	"ShortName",	"Size",	"Profitability",	"Volatility",	"TradingActivity",	"Leverage",	"Momentum",	"Value",	"Growth", "Return1Yr"]]

,Ticker,ShortName,Size,Profitability,Volatility,TradingActivity,Leverage,Momentum,Value,Growth,Return1Yr
4,AAL,AMERICAN AIRLINE,-0.872789,-0.884676,-0.961996,3.000228,2.427836,-0.610635,2.050937,-0.812962,8.188976
5,AAPL,APPLE INC,1.973204,2.738760,0.495583,-0.478149,0.085498,-0.130112,-0.372220,0.031300,49.371970
8,ABBV,ABBVIE INC,0.496940,1.530953,0.124997,-0.728043,0.839538,-0.148440,0.063431,-0.557802,-0.810278
11,ABNB,AIRBNB INC-A,-0.819412,0.108318,-0.543316,1.690136,-0.697833,-0.036284,-0.447422,1.276618,59.732482
12,ABT,ABBOTT LABS,0.098690,-0.457890,-0.388962,-0.784018,-0.477903,-0.687410,-0.191666,-0.600183,1.781966
...,...,...,...,...,...,...,...,...,...,...,...
684,WMT,WALMART INC,1.390133,-0.755175,-0.034558,-0.883312,-0.200955,0.060899,0.007342,-0.478348,12.599971
687,XEL,XCEL ENERGY INC,-0.731118,-0.785769,-0.226984,-0.110820,1.115379,-1.180554,0.434117,-0.838571,-9.926571
690,XOM,EXXON MOBIL CORP,1.375566,-0.080958,0.202097,-0.457229,-0.826035,0.131019,0.967992,-0.498213,-5.299010
691,XYZ,BLOCK INC,-0.932959,-1.413631,-0.066629,3.000228,-0.566332,-1.311343,-0.249530,1.817979,22.933885


In [18]:
df_2024.describe()

,Size,Profitability,Volatility,TradingActivity,Leverage,Momentum,Value,Growth,DividendYield,EarningsVariation,PriceToSales,EstPriceToSales,PriceToBook,EstPriceToBook,PriceToEarnings,EstPriceToEarnings,ReturnOnEquity,Sector,Industry,SubIndustry,ESGScore,Beta,Return1Yr,NumberOfEmployees,NumberOfEmployees5YearsAgo
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,2.850000e+02,284.000000,2.690000e+02,268.000000,275.000000,284.000000,265.000000,288.000000,288.000000,2.880000e+02,285.000000,297.000000,296.000000,2.820000e+02,2.750000e+02
mean,-0.256936,-0.398760,-0.120091,0.003761,0.286881,-0.248938,0.432433,-0.366054,0.460788,-0.213425,3.333612e+00,3.233990,7.481286e+00,5.272274,29.690311,17.938492,27.700041,33.055556,332338.975694,3.323392e+07,59.346354,0.698053,17.322069,8.667503e+04,7.666286e+04
std,0.618966,0.822383,0.679813,0.823292,0.818269,0.789721,0.819419,0.804089,0.957027,0.834400,3.574836e+00,3.028185,1.935655e+01,8.776850,94.961121,10.527929,76.450272,13.272947,132796.769058,1.327967e+07,11.124894,8.376645,36.218545,1.770887e+05,1.620598e+05
min,-0.995107,-3.000278,-2.301204,-0.989573,-1.381528,-3.000094,-3.000008,-2.159863,-1.077917,-1.099723,8.116883e-07,0.105223,1.926285e-08,0.364793,2.417327,3.858827,-78.914246,10.000000,101010.000000,1.010102e+07,19.897394,-123.384805,-99.000000,5.000000e+00,3.000000e+00
25%,-0.746991,-0.876446,-0.455339,-0.479537,-0.325458,-0.687410,-0.142042,-0.887097,-0.241258,-0.858323,1.198498e+00,1.308564,1.640124e+00,1.509114,11.266626,10.747454,10.021650,20.000000,203035.000000,2.030351e+07,52.801529,0.715129,-2.342056,1.827500e+04,1.430000e+04
50%,-0.377920,-0.486944,-0.105274,-0.222741,0.204379,-0.241731,0.217729,-0.569446,0.276729,-0.503253,2.087949e+00,2.352398,2.796241e+00,2.468004,17.841822,15.705487,16.151398,35.000000,352010.000000,3.520101e+07,60.004025,0.934107,11.257431,4.392300e+04,3.648400e+04
75%,0.088826,0.000000,0.180620,0.154117,0.874665,0.209769,0.811163,0.000000,1.021884,0.148771,4.223761e+00,3.983458,6.353123e+00,5.131749,25.816680,21.690061,27.757156,40.000000,403010.000000,4.030102e+07,67.458000,1.140786,26.272791,8.904850e+04,8.090000e+04
max,1.973204,2.881678,2.342157,3.000330,2.427836,3.000132,3.000015,3.000005,3.000010,3.000093,2.725291e+01,15.627382,2.413733e+02,91.445468,1437.913168,72.384421,1151.319702,60.000000,602010.000000,6.020104e+07,85.434059,72.385598,239.274541,2.100000e+06,2.300000e+06


In [19]:
df_2024.sort_values(by="Volatility", ascending=True).head(10)

,Ticker,ShortName,Size,Profitability,Volatility,TradingActivity,Leverage,Momentum,Value,Growth,DividendYield,EarningsVariation,PriceToSales,EstPriceToSales,PriceToBook,EstPriceToBook,PriceToEarnings,EstPriceToEarnings,ReturnOnEquity,RnDToSales,Sector,Industry,SubIndustry,SectorName,IndustryName,SubIndustryName,ESGScore,Beta,Return1Yr,NumberOfEmployees,NumberOfEmployees5YearsAgo
454,MU,MICRON TECH,-0.380408,-1.386005,-2.301204,1.686383,-0.397905,0.440698,-0.404964,1.088885,-0.416592,0.345225,5.776736,3.684479,2.184990,2.085782,NaN,43.185004,-14.908180,11.937169,45.0,453010.0,45301020.0,Information Technology,Semiconductors & Semiconductor Equipment,Semiconductors,63.796398,1.232522,70.037281,43000.0,36000.0
119,CAH,CARDINAL HEALTH,-0.364561,-1.182482,-2.065425,0.557005,0.536164,0.446375,0.094259,0.922200,0.298332,0.791167,0.122213,0.105223,NaN,NaN,12.609689,13.704304,NaN,0,35.0,351020.0,35102010.0,Health Care,Health Care Providers & Services,Health Care Distributors,59.561413,0.659773,33.510477,47040.0,50200.0
65,AZO,AUTOZONE INC,-0.964473,0.188525,-1.646327,0.327311,1.544852,0.039778,-0.090397,-0.089951,-0.722812,-0.169846,2.662459,2.356317,NaN,NaN,18.767585,16.817097,NaN,NaN,25.0,255040.0,25504050.0,Consumer Discretionary,Specialty Retail,Automotive Retail,55.416962,0.722825,4.919299,71400.0,54900.0
502,PCG,PG&E CORP,-0.452164,-0.925849,-1.633861,0.279283,1.613480,0.322913,0.591981,-0.455829,-0.722812,0.267317,1.605463,1.556669,1.485155,1.381260,14.773264,13.355556,7.919596,NaN,55.0,551010.0,55101010.0,Utilities,Electric Utilities,Electric Utilities,67.166283,0.694938,10.403780,10.0,23000.0
105,BNS,BANK OF NOVA SCO,0.501197,-1.009744,-1.563583,-0.357506,-0.960169,-0.625248,0.483818,-0.821158,2.528946,-0.890482,1.090242,2.299149,1.137291,1.081291,10.958598,9.814938,10.434095,NaN,40.0,401010.0,40101010.0,Financials,Banks,Diversified Banks,56.392715,1.021457,5.543409,89483.0,97629.0
495,ORLY,O'REILLY AUTOMOT,-0.976754,0.427109,-1.559962,0.067214,1.203269,0.417605,-0.205464,-0.004848,-0.722812,0.125093,3.724850,3.352287,NaN,NaN,25.274807,22.258980,NaN,NaN,25.0,255040.0,25504050.0,Consumer Discretionary,Specialty Retail,Automotive Retail,46.333366,0.599067,12.614092,87745.0,45440.0
498,OXY,OCCIDENTAL PETE,-0.348333,-0.037208,-1.533463,1.949072,0.369330,-0.633844,1.348544,-0.181526,-0.179088,0.656377,1.824577,1.689468,2.785858,2.145272,13.175230,11.523112,23.674873,0,10.0,101020.0,10102010.0,Energy,"Oil, Gas & Consumable Fuels",Integrated Oil & Gas,70.988831,0.798361,-2.974148,11973.0,11000.0
162,COR,CENCORA INC,-0.043906,0.682174,-1.489687,-0.057942,0.161707,0.291540,0.186928,1.115700,-0.241258,0.829328,0.158613,0.141979,79.009781,22.908188,19.327647,15.545859,NaN,0,35.0,351020.0,35102010.0,Health Care,Health Care Providers & Services,Health Care Distributors,56.543606,0.469938,25.068264,46000.0,20000.0
617,TD,TORONTO-DOM BANK,0.905939,-0.948231,-1.426865,-0.479537,-1.084829,-0.517382,-0.118421,-0.859420,1.494274,-0.947067,1.541291,3.015338,1.514209,1.400221,11.360937,10.735469,10.021650,NaN,40.0,401010.0,40101010.0,Financials,Banks,Diversified Banks,62.428326,0.938697,3.560005,103257.0,86588.0
539,REGN,REGENERON PHARM,-0.617505,-0.065605,-1.425192,-0.215812,-1.026233,-0.005110,-0.103921,0.525941,-0.722812,1.155375,7.164799,6.995486,3.836991,3.241473,22.209010,20.009522,17.224102,36.135756,35.0,352010.0,35201010.0,Health Care,Biotechnology,Biotechnology,61.351978,0.671045,21.766558,11851.0,6200.0


In [25]:
df_2024.columns

Index(['Ticker', 'ShortName', 'Size', 'Profitability', 'Volatility',
       'TradingActivity', 'Leverage', 'Momentum', 'Value', 'Growth',
       'DividendYield', 'EarningsVariation', 'PriceToSales', 'EstPriceToSales',
       'PriceToBook', 'EstPriceToBook', 'PriceToEarnings',
       'EstPriceToEarnings', 'ReturnOnEquity', 'RnDToSales', 'Sector',
       'Industry', 'SubIndustry', 'SectorName', 'IndustryName',
       'SubIndustryName', 'ESGScore', 'Beta', 'Return1Yr', 'NumberOfEmployees',
       'NumberOfEmployees5YearsAgo'],
      dtype='object')

In [20]:
df_2023.loc[df_2023.Ticker.map(len)<=4]

,Ticker,ShortName,Size,Profitability,Volatility,TradingActivity,Leverage,Momentum,Value,Growth,DividendYield,EarningsVariation,PriceToSales,EstPriceToSales,PriceToBook,EstPriceToBook,PriceToEarnings,EstPriceToEarnings,ReturnOnEquity,RnDToSales,Sector,Industry,SubIndustry,SectorName,IndustryName,SubIndustryName,ESGScore,Beta,Return1Yr,NumberOfEmployees,NumberOfEmployees5YearsAgo
6,AAL,AMERICAN AIRLINE,-0.719767,-1.341221,-1.105211,3.000263,2.426632,-0.375821,-0.823127,-0.120481,-0.817272,0.146493,0.182858,0.163673,NaN,NaN,NaN,8.579747,NaN,NaN,20.0,203020.0,20302010.0,Industrials,Passenger Airlines,Passenger Airlines,63.524796,1.366619,-29.607081,123400.0,122300.0
7,AAPL,APPLE INC,1.957850,2.593115,0.374834,-0.259648,0.167017,-0.259431,-0.242191,0.108389,-0.463189,0.372328,5.348388,5.030219,40.881141,32.346441,21.265138,20.483301,175.459290,8.071053,45.0,452020.0,45202030.0,Information Technology,"Technology Hardware, Storage & Peripherals","Technology Hardware, Storage & Peripherals",57.841263,1.204978,-26.664348,164000.0,123000.0
11,ABBV,ABBVIE INC,0.682031,1.660152,0.295544,-0.605930,0.745129,1.082947,-0.080965,-0.224200,0.952905,-0.392500,4.954442,5.260390,17.868656,10.507730,15.813123,13.826323,90.346603,22.705648,35.0,352010.0,35201010.0,Health Care,Biotechnology,Biotechnology,68.735542,0.565296,23.556542,50000.0,30000.0
16,ABNB,AIRBNB INC-A,-0.869328,-0.247090,-1.233799,2.482911,-0.677831,-1.205130,-0.400091,0.640439,-0.817272,2.095078,6.774357,5.879248,9.777071,6.638529,32.793513,28.543442,32.622395,18.519186,25.0,253010.0,25301020.0,Consumer Discretionary,"Hotels, Restaurants & Leisure","Hotels, Resorts & Cruise Lines",34.503571,1.497668,-49.342339,6132.0,NaN
17,ABT,ABBOTT LABS,0.312303,-0.109193,-0.339526,-0.696387,-0.493004,-0.216173,-0.221073,-0.725278,0.100462,-0.229170,4.298149,4.862459,5.365857,4.930816,23.473882,24.941718,22.508808,6.779499,35.0,351010.0,35101010.0,Health Care,Health Care Equipment & Supplies,Health Care Equipment,78.905540,0.887124,-20.834801,113000.0,75000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
926,WTW,WILLIS TOWERS WA,-0.894190,-0.446810,-0.761553,0.161237,-0.348832,0.330909,-0.110303,-0.689373,-0.159082,-0.705590,3.224881,2.893882,2.700995,2.950068,21.006516,16.254338,26.593189,NaN,40.0,403010.0,40301010.0,Financials,Insurance,Insurance Brokers,36.535561,0.877496,4.693283,44000.0,41000.0
927,XEL,XCEL ENERGY INC,-0.483149,-0.740004,-0.203117,-0.184181,0.898300,0.427947,0.067726,-0.700532,0.523147,-0.927662,2.615653,2.699165,2.340735,2.203627,22.911765,20.776851,10.597370,NaN,55.0,551010.0,55101010.0,Utilities,Electric Utilities,Electric Utilities,71.833817,0.657713,5.641302,11321.0,11075.0
930,XOM,EXXON MOBIL CORP,1.486901,-0.118543,0.061943,-0.215250,-0.833794,1.782989,0.923906,-0.866065,0.811943,0.411162,1.207891,1.202259,2.440706,2.119010,8.719355,9.793259,29.917303,NaN,10.0,101020.0,10102010.0,Energy,"Oil, Gas & Consumable Fuels",Integrated Oil & Gas,67.618950,0.690480,88.588112,63000.0,71100.0
931,XYZ,BLOCK INC,-0.678655,-1.298511,0.307278,3.000263,-0.637994,-1.498321,-0.497665,2.561815,-0.817272,0.935745,2.005896,1.880089,2.272224,2.685771,NaN,37.005205,-5.177986,12.082453,40.0,402010.0,40201060.0,Financials,Financial Services,Transaction & Payment Processing Services,46.454079,2.204972,-61.993467,8521.0,1853.0


In [21]:
prc_df_2023 = pd.read_parquet('data/original_data/prc_df_2023.parquet')
prc_df_2023 = prc_df_2023.dropna(axis=1)


In [22]:
prc_df_2023

Ticker,AAL,AAPL,ABBV,ABNB,ABT,ACGL,ACI,ACN,ADBE,ADI,ADM,ADP,AEE,AEP,AER,AES,AFL,AGR,AIG,AJG,ALC,ALL,ALLY,AMAT,AMD,AMGN,AMP,AMT,AMZN,AON,APD,APH,APO,ASML,AVGO,AXP,AZO,BA,BAC,BAX,BBY,BCE,BDX,BHF,BIIB,BK,BKNG,BKR,BMO,BMY,BN,BNS,BSX,BX,C,CAH,CARR,CAT,CB,CBRE,CCEP,CCI,CDW,CEG,CFG,CHTR,CI,CL,CM,CME,CMI,CNC,CNH,CNI,CNP,CNQ,COF,COP,COR,COST,CP,CRH,CRM,CSCO,CSX,CTSH,CTVA,CVE,CVS,CVX,D,DAL,DB,DD,DE,DELL,DFS,DG,DHI,DHR,DIS,DLR,DLTR,DOW,DTE,DUK,DVN,ECL,ED,EG,EIX,EL,ELV,EMR,ENB,EOG,EPD,EQH,EQIX,ES,ET,ETN,ETR,EXC,F,FANG,FCX,FDX,FE,FERN,FI,FIS,FITB,FNF,FNRC,FRCB,FTS,GD,GE,GILD,GIS,GLW,GM,GOLD,GOOG,GPC,GPN,GS,HAL,HBAN,HCA,HD,HES,HIG,HON,HPE,HPQ,HSY,HUM,IBKR,IBM,ICE,IFF,IMO,INTC,INTU,IQV,ISRG,ITW,JCI,JNJ,JPM,JXN,K,KDP,KEY,KHC,KKR,KLAC,KMB,KMI,KMX,KO,KR,L,LEN,LHX,LLY,LMT,LNC,LNG,LOW,LRCX,LUV,LYB,LYG,MA,MAR,MCD,MCHP,MCK,MCO,MDLZ,MDT,MELI,MET,META,MFC,MGA,MGM,MKL,MMC,MMM,MO,MOH,MOS,MPC,MPLX,MRK,MRNA,MS,MSFT,MT,MTB,MU,NEE,NEM,NFLX,NKE,NOC,NOW,NSC,NTR,NTRS,NUE,NVDA,NXPI,O,OKE,ORCL,ORLY,OXY,PAA,PARA,PCAR,PCG,PDD,PEG,PEP,PFE,PFG,PG,PGR,PH,PHG,PLD,PM,PNC,PPG,PPL,PRU,PSX,PYPL,QCOM,REGN,RF,RGA,RJF,ROP,ROST,RSG,RTX,RY,SBUX,SCCO,SCHW,SHW,SLB,SLF,SNX,SO,SPG,SPGI,SPRV,SRE,STLA,STM,STT,STZ,SU,SVXY,SYF,SYK,SYY,T,TD,TECK,TFC,TGT,TJX,TMO,TMUS,TRGP,TRP,TRV,TSLA,TSN,TT,TU,TXN,UAL,UBER,UNH,UNP,UPS,URI,USB,UVXY,V,VIXM,VIXY,VLO,VRTX,VTRS,VXX,VXZ,VZ,WBA,WBD,WEC,WELL,WFC,WM,WMB,WMT,WRB,WTW,XEL,XOM,XYZ,ZTS
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-01-03,12.74,123.632523,149.164108,84.900002,104.779053,59.393120,19.611376,261.824371,336.920013,156.755081,83.634407,227.386276,82.914330,86.777695,57.485336,25.735664,68.086044,39.008400,60.120926,183.706192,68.536598,129.909256,22.488546,94.880524,64.019997,243.500778,300.683563,200.793762,85.820000,294.697968,291.319366,37.503994,60.327297,539.838257,53.397110,142.924423,2431.060059,195.389999,31.691650,48.181396,73.375404,37.612625,248.158539,51.040001,272.630005,43.170719,2014.349731,27.223515,82.620468,65.689964,31.083275,42.613499,46.009998,71.071815,41.898056,73.629539,40.473133,229.551056,214.210449,78.440002,51.223434,123.322502,174.761917,80.366318,35.463558,341.579987,310.605743,75.046204,36.710869,153.949326,227.850983,79.739998,15.041700,115.038155,27.779707,23.747482,89.124313,105.279388,160.675415,436.833771,73.958015,38.840900,134.031387,44.742989,30.033775,55.511196,57.571247,17.394024,85.305573,158.737137,56.787544,32.027203,11.027692,66.211334,412.431000,38.959743,92.037918,237.219025,88.852730,230.468842,87.884308,95.622894,140.259995,45.894928,109.783836,94.552734,52.896431,144.011230,88.496017,322.539856,59.005463,244.567307,489.243286,91.732132,34.593353,114.444595,20.580132,27.092442,633.336914,77.240532,9.589602,153.045151,46.888412,39.623119,9.566102,119.176521,36.758308,170.126160,38.218536,0.00140,101.209999,64.903152,29.949360,34.804111,0.0001,121.299080,37.304901,236.807236,52.438713,78.986496,77.185219,30.512453,33.117290,17.100534,89.378845,160.833694,98.420326,327.647125,36.224506,12.770592,239.135498,300.008942,131.113815,72.287659,205.286530,15.163097,24.961742,214.306854,491.160400,70.599174,129.286346,100.416870,99.560631,44.127991,25.775145,385.398315,203.860001,265.630005,210.635574,61.830021,166.303711,127.490326,31.245569,62.280090,33.548054,15.624210,37.219692,45.816761,367.926422,127.417076,15.775637,60.639999,59.216160,42.229450,58.123062,86.386391,197.872818,358.463654,453.063049,26.606739,137.336670,190.490509,37.085289,30.993149,75.341171,2.006593,342.209991,144.444031,252.362473,65.765678,370.442841,271.165344,63.049377,73.202179,826.109985,67.221916,124.265312,16.459894,53.372166,33.279999,1321.969971,160.665405,92.868958,38.414948,318.920013,40.331993,105.604301,26.856091

In [23]:
returns = prc_df_2023.pct_change().dropna()

In [24]:
returns

Ticker,AAL,AAPL,ABBV,ABNB,ABT,ACGL,ACI,ACN,ADBE,ADI,ADM,ADP,AEE,AEP,AER,AES,AFL,AGR,AIG,AJG,ALC,ALL,ALLY,AMAT,AMD,AMGN,AMP,AMT,AMZN,AON,APD,APH,APO,ASML,AVGO,AXP,AZO,BA,BAC,BAX,BBY,BCE,BDX,BHF,BIIB,BK,BKNG,BKR,BMO,BMY,BN,BNS,BSX,BX,C,CAH,CARR,CAT,CB,CBRE,CCEP,CCI,CDW,CEG,CFG,CHTR,CI,CL,CM,CME,CMI,CNC,CNH,CNI,CNP,CNQ,COF,COP,COR,COST,CP,CRH,CRM,CSCO,CSX,CTSH,CTVA,CVE,CVS,CVX,D,DAL,DB,DD,DE,DELL,DFS,DG,DHI,DHR,DIS,DLR,DLTR,DOW,DTE,DUK,DVN,ECL,ED,EG,EIX,EL,ELV,EMR,ENB,EOG,EPD,EQH,EQIX,ES,ET,ETN,ETR,EXC,F,FANG,FCX,FDX,FE,FERN,FI,FIS,FITB,FNF,FNRC,FRCB,FTS,GD,GE,GILD,GIS,GLW,GM,GOLD,GOOG,GPC,GPN,GS,HAL,HBAN,HCA,HD,HES,HIG,HON,HPE,HPQ,HSY,HUM,IBKR,IBM,ICE,IFF,IMO,INTC,INTU,IQV,ISRG,ITW,JCI,JNJ,JPM,JXN,K,KDP,KEY,KHC,KKR,KLAC,KMB,KMI,KMX,KO,KR,L,LEN,LHX,LLY,LMT,LNC,LNG,LOW,LRCX,LUV,LYB,LYG,MA,MAR,MCD,MCHP,MCK,MCO,MDLZ,MDT,MELI,MET,META,MFC,MGA,MGM,MKL,MMC,MMM,MO,MOH,MOS,MPC,MPLX,MRK,MRNA,MS,MSFT,MT,MTB,MU,NEE,NEM,NFLX,NKE,NOC,NOW,NSC,NTR,NTRS,NUE,NVDA,NXPI,O,OKE,ORCL,ORLY,OXY,PAA,PARA,PCAR,PCG,PDD,PEG,PEP,PFE,PFG,PG,PGR,PH,PHG,PLD,PM,PNC,PPG,PPL,PRU,PSX,PYPL,QCOM,REGN,RF,RGA,RJF,ROP,ROST,RSG,RTX,RY,SBUX,SCCO,SCHW,SHW,SLB,SLF,SNX,SO,SPG,SPGI,SPRV,SRE,STLA,STM,STT,STZ,SU,SVXY,SYF,SYK,SYY,T,TD,TECK,TFC,TGT,TJX,TMO,TMUS,TRGP,TRP,TRV,TSLA,TSN,TT,TU,TXN,UAL,UBER,UNH,UNP,UPS,URI,USB,UVXY,V,VIXM,VIXY,VLO,VRTX,VTRS,VXX,VXZ,VZ,WBA,WBD,WEC,WELL,WFC,WM,WMB,WMT,WRB,WTW,XEL,XOM,XYZ,ZTS
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-01-04,0.066719,0.010314,0.008067,0.044994,0.014875,0.004963,0.009662,-0.003404,0.013327,0.021299,-0.036364,0.004713,0.005311,0.006851,0.026402,-0.029527,0.004750,0.013444,0.014778,0.013244,0.018701,0.016830,0.033579,0.026672,0.009997,0.010472,0.007146,0.020170,-0.007924,0.011021,0.000065,0.014790,0.024678,0.038685,0.012214,0.023246,0.005923,0.042223,0.018801,0.025274,0.019027,0.019746,0.007456,0.021552,-0.006676,0.024622,0.038495,-0.029913,0.028449,0.004982,0.042459,0.032935,0.010650,0.024694,0.025776,0.003519,0.010094,0.010382,0.016442,0.016318,0.023915,0.036440,0.009136,0.018313,0.028333,0.036360,-0.034777,0.006445,0.024679,0.006689,-0.002406,-0.003511,0.003724,0.015188,0.003045,0.002301,0.030440,0.001945,-0.008643,0.007236,0.024172,0.048014,0.035688,-0.000210,0.012953,0.008161,0.012897,-0.006575,-0.010010,-0.010633,0.003811,0.054584,0.065756,0.024252,-0.001909,0.000978,0.044128,-0.013051,0.010587,0.015633,0.033832,0.007974,0.001854,0.025274,0.014424,0.010704,0.013421,0.017322,0.008667,0.023127,0.032825,0.031507,-0.033373,-0.007179,0.018960,-0.000723,0.004540,0.037413,0.026626,0.013180,0.003442,0.006783,0.000733,0.013207,0.028254,-0.005785,0.028481,0.023580,0.007376,0.035714,0.008398,0.010651,0.025123,0.019571,0.0,0.018512,0.011111,-0.013596,0.058204,0.001288,0.008051,0.048249,0.025724,0.058889,-0.011037,0.012544,0.032735,0.004275,0.020446,0.018440,0.023313,0.012092,0.015239,0.019509,-0.019879,0.022416,0.017570,-0.016128,-0.016724,0.010367,0.007418,0.006781,0.023855,-0.003243,0.035541,0.000997,0.019229,0.016451,0.002270,0.016373,0.010887,0.009325,0.015593,-0.007577,0.008157,0.026689,0.011280,0.030420,0.009773,-0.002188,0.012215,0.073879,-0.000477,0.011244,0.011606,0.017721,0.000000,-0.005178,-0.002157,0.021090,0.029602,0.011807,0.019696,0.031288,0.022741,0.022321,0.024078,0.014281,0.000227,0.024457,-0.000855,0.027634,0.007245,0.034628,0.013715,0.014414,0.021084,0.025252,0.033727,0.067909,0.019690,0.019460,0.021883,-0.002416,-0.016901,0.015013,-0.003065,0.007366,0.008458,-0.009107,0.010849,-0.043743,0.031919,0.011175,0.076037,0.007873,0.038927,0.049025,0.020716,-0.025688,0.021660,0.012919,0.005027,0.021163,0.045323,0.030318,0.018768,0.012696,0.007820,0.009078,-0.016629,0.000983,0.000000,0.083772,-0.002235,0.012755,0.07725

## 2024

In [12]:
prc_df_2024 = pd.read_parquet('data/original_data/prc_df_2024.parquet')
prc_df_2024 = prc_df_2024.dropna(axis=1)


In [13]:
prc_df_2024

Ticker,AAL,AAPL,ABBV,ABNB,ABT,ACGL,ACI,ACN,ADBE,ADI,...,WBD,WELL,WFC,WM,WMB,WMT,XEL,XOM,XYZ,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,13.44,184.532089,152.743393,134.479996,107.138489,71.840385,22.120523,341.431366,580.070007,190.350952,...,11.66,87.767197,47.823627,177.070602,33.995880,52.467777,61.121792,98.105995,72.220001,194.055588
2024-01-03,12.95,183.150391,153.355057,133.419998,106.816628,73.162132,22.004251,332.573761,571.789978,185.808029,...,11.30,86.473648,47.193474,176.005859,34.751980,52.471069,61.227722,98.930260,68.629997,190.462128
2024-01-04,13.09,180.824341,154.310791,133.720001,108.240601,73.666107,21.839539,331.756866,567.049988,182.966232,...,11.35,86.269402,47.775150,176.262207,34.283009,51.963818,61.372169,98.067657,68.150002,191.557953
2024-01-05,13.60,180.098709,154.960678,135.979996,108.065041,73.038513,22.323997,331.294312,564.599976,183.438232,...,11.20,86.094337,48.395611,175.256622,34.024590,51.617966,61.372169,98.364784,66.959999,192.357574
2024-01-08,14.58,184.452576,154.282104,140.080002,109.625542,73.333298,22.275551,334.965271,580.549988,185.827698,...,11.40,87.348984,48.395611,175.719986,34.120300,52.125221,61.227722,96.725830,69.309998,193.640945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,15.31,247.100006,204.080002,144.820007,134.949997,89.550003,20.430000,363.910004,444.420013,236.990005,...,11.09,149.970001,77.220001,229.679993,57.330002,93.669998,70.779999,111.269997,65.919998,165.309998
2025-02-25,15.13,247.039993,204.139999,141.550003,135.820007,90.349998,20.650000,362.950012,443.410004,235.300003,...,10.69,151.740005,76.000000,230.750000,56.180000,97.690002,70.449997,109.730003,62.840000,168.529999
2025-02-26,15.18,240.360001,203.009995,144.029999,135.960007,89.570000,20.690001,358.350006,441.500000,233.729996,...,10.50,150.649994,76.449997,228.690002,56.900002,96.199997,69.970001,109.459999,65.290001,164.990005


In [14]:
returns_2024 = prc_df_2024.pct_change().dropna()

In [15]:
returns_2024

Ticker,AAL,AAPL,ABBV,ABNB,ABT,ACGL,ACI,ACN,ADBE,ADI,...,WBD,WELL,WFC,WM,WMB,WMT,XEL,XOM,XYZ,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-03,-0.036458,-0.007488,0.004005,-0.007882,-0.003004,0.018398,-0.005256,-0.025943,-0.014274,-0.023866,...,-0.030875,-0.014738,-0.013177,-0.006013,0.022241,0.000063,0.001733,0.008402,-0.049709,-0.018518
2024-01-04,0.010811,-0.012700,0.006232,0.002249,0.013331,0.006888,-0.007486,-0.002456,-0.008290,-0.015294,...,0.004425,-0.002362,0.012325,0.001456,-0.013495,-0.009667,0.002359,-0.008719,-0.006994,0.005754
2024-01-05,0.038961,-0.004013,0.004212,0.016901,-0.001622,-0.008519,0.022183,-0.001394,-0.004321,0.002580,...,-0.013216,-0.002029,0.012987,-0.005705,-0.007538,-0.006656,0.000000,0.003030,-0.017462,0.004174
2024-01-08,0.072059,0.024175,-0.004379,0.030152,0.014440,0.004036,-0.002170,0.011081,0.028250,0.013026,...,0.017857,0.014573,0.000000,0.002644,0.002813,0.009827,-0.002354,-0.016662,0.035096,0.006672
2024-01-09,-0.013717,-0.002263,0.005451,-0.003926,0.002936,-0.012578,0.004785,0.007052,0.009732,0.004127,...,-0.032456,-0.002450,-0.012620,-0.000056,-0.006452,0.006698,-0.004876,-0.012386,0.003174,-0.001071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,0.005253,0.006312,0.009897,-0.008558,0.000222,0.016113,-0.005839,-0.000961,0.000225,-0.008036,...,0.028757,0.013865,-0.003613,0.007236,-0.001915,-0.011711,0.012010,0.005240,-0.035552,0.030226
2025-02-25,-0.011757,-0.000243,0.000294,-0.022580,0.006447,0.008934,0.010768,-0.002638,-0.002273,-0.007131,...,-0.036069,0.011802,-0.015799,0.004659,-0.020059,0.042917,-0.004662,-0.013840,-0.046723,0.019479
2025-02-26,0.003305,-0.027040,-0.005535,0.017520,0.001031,-0.008633,0.001937,-0.012674,-0.004308,-0.006672,...,-0.017774,-0.007183,0.005921,-0.008927,0.012816,-0.015252,-0.006813,-0.002461,0.038988,-0.021005


In [13]:
test = pd.read_csv('data/submission1.csv')

In [14]:
test

,ID,WEIGHT
0,V,-0.028191
1,VXZ,-0.028078
2,ADI,-0.028023
3,SYF,-0.027975
4,AMZN,-0.027467
...,...,...
292,ALL,0.027636
293,ET,0.027835
294,HON,0.027951
295,ICE,0.028109


In [15]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      297 non-null    object 
 1   WEIGHT  297 non-null    float64
dtypes: float64(1), object(1)
memory usage: 4.8+ KB


In [16]:
test["WEIGHT"].sum()

1.0

In [17]:
submission = test.copy()
submission["WEIGHT"] = submission["WEIGHT"].apply(lambda x: int(x * 1000) / 1000)

In [18]:
submission["WEIGHT"].sum()

0.9659999999999997